# Workflow

* Ticker -> Look up CIK -> Fetch filing list -> Find latest 10-K -> Download document

## Setup

In [1]:
import os
from pathlib import Path
import requests

HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Look Up CIK

In [2]:
# Downloads the SEC's full ticker-to-CIK mapping
resp = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
resp.raise_for_status()

ticker_to_cik = {item["ticker"]: item["cik_str"] for item in resp.json().values()}

cik = ticker_to_cik["AAPL"]
print(cik)

320193


## Find the Latest 10-K

In [3]:
# This endpoint needs a 10-digit CIK
url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
resp = requests.get(url, headers=HEADERS)
resp.raise_for_status()

recent = resp.json()["filings"]["recent"]
idx = recent["form"].index("10-K")

filing_date = recent["filingDate"][idx]
# Unique ID for each filing, used to build the document URL
accession = recent["accessionNumber"][idx]
primary_doc = recent["primaryDocument"][idx]
print(filing_date, accession, primary_doc)

2026-11-03 0000320193-26-000026 aapl-20260926.htm


## Download the Document

In [4]:
# The actual 10-K file URL
doc_url = (
	f"https://www.sec.gov/Archives/edgar/data/{cik}/"
	f"{accession.replace('-', '')}/{primary_doc}"
)
resp = requests.get(doc_url, headers=HEADERS)
resp.raise_for_status()

# Saves as TICKER_FORM_DATE.html, keeping files easy to identify later
out_path = DATA_DIR / f"AAPL_10-K_{filing_date}.html"
out_path.write_text(resp.text, encoding="utf-8")
print(out_path, f"{len(resp.text):,} characters")

../data/raw/AAPL_10-K_2026-11-03.html 1,650,108 characters
